# EXAMEN 3 — Fundamentos de Data Science
## Analizando la Satisfacción de Clientes en la Industria Hotelera — **SOLUCIÓN**

---

**Dataset:** `hotel_bookings.csv` — reservas de dos tipos de hotel (City Hotel y Resort Hotel), con información sobre cancelaciones, estancias, segmento de mercado, tarifas y país de origen del cliente.

> Este notebook contiene la solución de referencia del Examen 3.

---
## Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

df = pd.read_csv('../../hotel_bookings.csv')

print(f'Dataset cargado — shape: {df.shape}')
print(f'Columnas ({len(df.columns)}): {list(df.columns)}')

---
## 1. Exploración inicial

In [ ]:
display(df.head())
display(df.tail())
df.info()
display(df.describe())

**Observaciones iniciales:**

- 119,390 reservas con 32 columnas. Mezcla de numéricas (tarifas, noches, anticipación), categóricas (hotel, comida, país, segmento) y fechas.
- `reservation_status_date` está como `object` — debería ser `datetime` para análisis temporal.
- `agent` y `company` son `float64` con muchos `NaN` (códigos de agencia/empresa, no son métricas).
- `children` es `float64` por los pocos `NaN` — debería ser entero después de imputar.
- Variables que requieren atención: `adr` (tarifa) puede tener valores negativos o extremos; `adults+children+babies` puede dar 0 (reservas fantasma).
- Hay valores tipo string `'NULL'` en columnas como `country` que deben tratarse como nulos.

---
## 2. Limpieza de datos

### 2.1 Detección y eliminación de duplicados

In [ ]:
n_duplicados = df.duplicated().sum()
print(f'Filas duplicadas encontradas: {n_duplicados:,}')
print(f'Porcentaje del dataset: {n_duplicados/len(df)*100:.2f}%')

if n_duplicados > 0:
    print('\nEjemplos de filas duplicadas:')
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6))

shape_antes = df.shape
df = df.drop_duplicates().reset_index(drop=True)
shape_despues = df.shape

print(f'\nShape antes:   {shape_antes}')
print(f'Shape después: {shape_despues}')
print(f'Filas eliminadas: {shape_antes[0] - shape_despues[0]:,}')

**Observación:** ~32,000 filas duplicadas (≈27% del dataset). Es un volumen muy alto que afecta cualquier estadística agregada — si no se eliminan, conteos por país, hotel o segmento aparecerán inflados. `drop_duplicates()` los remueve quedándonos con la primera ocurrencia.

### 2.2 Verificación y ajuste de tipos de datos

In [ ]:
print('Tipos ANTES de corregir:')
print(df.dtypes)
print()

# reservation_status_date → datetime para poder operar temporalmente
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'], errors='coerce')

# Construir arrival_date como datetime real combinando year/month/day
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str),
    errors='coerce'
)

# Convertir 'NULL' (string) a NaN real en columnas conocidas
for col in ['country', 'agent', 'company']:
    df[col] = df[col].replace('NULL', np.nan)

# agent y company son códigos identificadores, no métricas — los convertimos a string
# para evitar que aparezcan en describe() como si fueran numéricos
df['agent']   = pd.to_numeric(df['agent'],   errors='coerce')
df['company'] = pd.to_numeric(df['company'], errors='coerce')

print('Tipos DESPUÉS de corregir:')
print(df.dtypes)

**Observación:**

- `reservation_status_date` ahora es `datetime64` — permite extraer año/mes/día y hacer análisis temporales.
- Creamos `arrival_date` combinando las tres columnas de fecha de llegada — más cómodo que tener tres columnas separadas.
- `'NULL'` (string) en `country`, `agent` y `company` se convierte a `NaN` real con `pd.to_numeric(errors='coerce')` y `replace`.
- `agent` y `company` son códigos identificadores; los mantenemos numéricos pero los trataremos como categóricos al analizar.

### 2.3 Consistencia en valores categóricos

In [ ]:
# Revisar valores únicos antes de limpiar
cols_cat = ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel',
            'reserved_room_type', 'assigned_room_type', 'deposit_type',
            'customer_type', 'reservation_status']

for col in cols_cat:
    print(f'{col}: {df[col].nunique()} valores únicos — ejemplos: {sorted(df[col].dropna().unique())[:6]}')

In [ ]:
# Normalizar texto: quitar espacios y unificar capitalización
for col in cols_cat:
    df[col] = df[col].astype(str).str.strip()

# 'meal' tiene la categoría 'Undefined' que en realidad significa sin información — la unificamos con SC
# (Self Catering / sin comida), que es el comportamiento documentado del dataset
df['meal'] = df['meal'].replace({'Undefined': 'SC'})

# 'distribution_channel' y 'market_segment' tienen también 'Undefined' — lo dejamos como categoría propia
# pero asegurándonos de que esté escrito consistentemente
for col in ['distribution_channel', 'market_segment']:
    df[col] = df[col].replace({'undefined': 'Undefined'})

print('Categorías después de normalizar:')
for col in ['hotel', 'meal', 'market_segment', 'distribution_channel', 'customer_type']:
    print(f'\n{col}: {sorted(df[col].unique())}')

**Observación:**

- Se aplica `.str.strip()` a todas las columnas categóricas para eliminar espacios accidentales.
- `'meal' = 'Undefined'` se unifica con `'SC'` (Self Catering / sin comida) — son semánticamente lo mismo según la documentación del dataset.
- En `distribution_channel` y `market_segment` `'Undefined'` es una categoría legítima (canal desconocido), así que la dejamos como categoría propia, solo asegurando consistencia ortográfica.
- Las columnas de tipo de habitación (`reserved_room_type`, `assigned_room_type`) ya están normalizadas como letras únicas.

### 2.4 Manejo de valores faltantes

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': nulos_pct
}).query('Nulos > 0').sort_values('Porcentaje (%)', ascending=False)

print('Columnas con valores faltantes:')
display(resumen_nulos)

In [ ]:
# Estrategia de imputación por tipo de columna

# children: pocos nulos → 0 (asumimos que no había niños registrados)
df['children'] = df['children'].fillna(0).astype(int)
print(f'children → rellenado con 0 y convertido a int')

# country: ~500 nulos → 'Unknown' (no podemos inferir el país de origen)
df['country'] = df['country'].fillna('Unknown')
print(f'country → rellenado con "Unknown"')

# agent: ~16k nulos → 0 (sin agencia intermediaria, reserva directa)
df['agent'] = df['agent'].fillna(0).astype(int)
print(f'agent → rellenado con 0 (sin agencia)')

# company: ~112k nulos (94% del dataset) → 0 (sin empresa asociada)
# Es tan masivo que cualquier imputación sería ruido — el 0 es semánticamente correcto (cliente sin empresa)
df['company'] = df['company'].fillna(0).astype(int)
print(f'company → rellenado con 0 (sin empresa)')

print(f'\nNulos restantes: {df.isnull().sum().sum()}')

**Observaciones:**

- **`children` → 0**: Solo 4 nulos. Lo más probable es que no se haya registrado porque no había niños. Imputar con 0 mantiene el significado.
- **`country` → 'Unknown'`**: Tenemos ~500 nulos. No podemos inferir el país, así que un marcador explícito es más honesto que usar la moda (Portugal) que distorsionaría el análisis geográfico.
- **`agent` → 0**: ~16k nulos significan reservas sin agencia intermediaria. El 0 es el marcador convencional para 'sin agente'.
- **`company` → 0**: 94% de nulos confirma que la mayoría de reservas no están asociadas a una empresa. El 0 indica 'cliente individual sin convenio corporativo'.
- No usamos la mediana ni la moda en ninguna de estas columnas porque son códigos identificadores, no métricas continuas.

### 2.5 Detección y corrección de datos anómalos

In [ ]:
print('=== Diagnóstico de datos anómalos ===\n')

# 1. Tarifa (adr) negativa o extremadamente alta
print(f'adr negativo: {(df["adr"] < 0).sum()} casos')
print(f'adr extremo (> 1000): {(df["adr"] > 1000).sum()} casos')
print(f'adr máximo: {df["adr"].max():.2f}')

# Corregir: ningún hotel cobra negativo. Recortamos a 0 los negativos.
# Para los extremos > 1000 los recortamos al p99.99 (preserva la cola alta legítima)
p_alto = df['adr'].quantile(0.9999)
df['adr'] = df['adr'].clip(lower=0, upper=p_alto)
print(f'  → recortado a [0, {p_alto:.2f}]\n')

# 2. Reservas fantasma: 0 adultos + 0 niños + 0 bebés (nadie se hospeda)
fantasmas = (df['adults'] + df['children'] + df['babies'] == 0).sum()
print(f'Reservas sin huéspedes (0 adultos + 0 niños + 0 bebés): {fantasmas}')
print('  → eliminadas (registros inválidos)')
df = df[(df['adults'] + df['children'] + df['babies']) > 0].reset_index(drop=True)

# 3. Estancias de 0 noches (ni semana ni fin de semana)
cero_noches = ((df['stays_in_week_nights'] + df['stays_in_weekend_nights']) == 0).sum()
print(f'\nReservas de 0 noches totales: {cero_noches}')
print('  → se conservan: pueden ser day-use (uso por horas), pero las marcamos')
df['total_nights'] = df['stays_in_week_nights'] + df['stays_in_weekend_nights']

# 4. Estancias absurdamente largas
largas = (df['total_nights'] > 30).sum()
print(f'\nReservas > 30 noches: {largas} casos')
print(f'  → estancias máximas: {df["total_nights"].max()} noches')
# No se eliminan: pueden ser legítimas (estancias largas corporativas)

print(f'\nDataset final tras limpieza: {df.shape[0]:,} filas × {df.shape[1]} columnas')

**Observaciones:**

- **`adr` negativo**: imposible — los hoteles no pagan al cliente. Lo recortamos a 0.
- **`adr` extremo (> 1000)**: hay un caso con valor de 5400 que es claramente un error de captura. Lo recortamos al percentil 99.99 para preservar la cola alta legítima de hoteles premium.
- **Reservas fantasma**: 0 adultos + 0 niños + 0 bebés es una reserva sin huéspedes, sin sentido. Se eliminan.
- **Reservas de 0 noches**: pueden ser day-use legítimas o cancelaciones inmediatas — las mantenemos pero creamos `total_nights` para detectarlas en el análisis.
- **Estancias largas (> 30 noches)**: las dejamos porque pueden ser reservas corporativas o de larga estadía legítimas, pero las identificamos.

### Resumen del dataset limpio

In [ ]:
print('=== DATASET LIMPIO ===')
df.info()
print()
print(f'Nulos restantes: {df.isnull().sum().sum()}')
display(df.describe().round(2))
display(df.head())

---
## 3. Visualización univariada #1 — Distribución de la tarifa diaria (adr)

In [ ]:
# Filtramos top 1% para que el histograma sea legible
p99 = df['adr'].quantile(0.99)
df_vis = df[df['adr'] <= p99]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de la tarifa diaria (ADR) — euros por noche', fontsize=14, fontweight='bold')

sns.histplot(df_vis['adr'], bins=50, kde=True, color='steelblue', alpha=0.75, ax=axes[0])
axes[0].axvline(df['adr'].mean(),   color='orange', linewidth=2, linestyle='--',
                label=f"Media: €{df['adr'].mean():.2f}")
axes[0].axvline(df['adr'].median(), color='red', linewidth=2, linestyle='-',
                label=f"Mediana: €{df['adr'].median():.2f}")
axes[0].set_title('Distribución global')
axes[0].set_xlabel('ADR (€)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Mismo histograma separado por tipo de hotel
for hotel, color in zip(['City Hotel', 'Resort Hotel'], ['#4472C4', '#ED7D31']):
    sub = df_vis[df_vis['hotel'] == hotel]['adr']
    axes[1].hist(sub, bins=40, alpha=0.55, label=hotel, color=color, edgecolor='white')
axes[1].set_title('Por tipo de hotel')
axes[1].set_xlabel('ADR (€)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Media:   €{df['adr'].mean():.2f}")
print(f"Mediana: €{df['adr'].median():.2f}")
print(f"Std:     €{df['adr'].std():.2f}")

**Interpretación:**

- La distribución de tarifas tiene **asimetría positiva**: la mayoría de reservas se concentra en el rango de €50–€120, pero hay una cola larga de tarifas premium.
- La mediana (~€95) es menor que la media (~€102), confirmando el sesgo.
- Comparando por tipo de hotel: **City Hotel** tiene tarifas más concentradas y consistentes; **Resort Hotel** tiene mayor dispersión, probablemente porque su tarifa varía mucho con la temporada (alta vs baja).

---
## 4. Visualización univariada #2 — Tasa de cancelación por tipo de hotel

In [ ]:
tasa_cancel = df.groupby('hotel')['is_canceled'].agg(['mean', 'count']).reset_index()
tasa_cancel.columns = ['hotel', 'tasa_cancelacion', 'reservas_totales']
tasa_cancel['tasa_cancelacion'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Cancelaciones por tipo de hotel', fontsize=14, fontweight='bold')

# Barplot — tasa de cancelación
colores = ['#4472C4', '#ED7D31']
bars = axes[0].bar(tasa_cancel['hotel'], tasa_cancel['tasa_cancelacion'],
                   color=colores, edgecolor='white', linewidth=2)
for bar, val in zip(bars, tasa_cancel['tasa_cancelacion']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Tasa de cancelación')
axes[0].set_ylabel('% de reservas canceladas')
axes[0].set_ylim(0, max(tasa_cancel['tasa_cancelacion']) * 1.2)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart — volumen de reservas
axes[1].pie(tasa_cancel['reservas_totales'],
            labels=tasa_cancel['hotel'],
            autopct='%1.1f%%',
            colors=colores,
            startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Distribución de reservas')

plt.tight_layout()
plt.show()

display(tasa_cancel.round(2))

**Interpretación:**

- **City Hotel cancela mucho más que Resort Hotel** — la diferencia es de varios puntos porcentuales. Coherente con el patrón típico del sector: las estancias urbanas suelen ser más cortas y flexibles, con cancelaciones de última hora habituales (viajes de negocio que se reprograman).
- City Hotel concentra la mayoría del volumen de reservas (~60-65%), lo que lo hace estratégicamente más importante en términos de ingresos absolutos a pesar de su mayor tasa de cancelación.
- **Insight de negocio:** el equipo de revenue management debería enfocarse en estrategias específicas de retención para City Hotel (políticas de prepago, descuentos por no reembolsable).

---
## 5. Visualización multivariada #3 — Estacionalidad de la tarifa por hotel

In [ ]:
orden_meses = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

adr_mes_hotel = (df.groupby(['arrival_date_month', 'hotel'])['adr']
                   .mean()
                   .reset_index())
adr_mes_hotel['arrival_date_month'] = pd.Categorical(
    adr_mes_hotel['arrival_date_month'], categories=orden_meses, ordered=True)
adr_mes_hotel = adr_mes_hotel.sort_values('arrival_date_month')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Estacionalidad de la tarifa promedio (ADR)', fontsize=14, fontweight='bold')

# Lineplot por hotel
for hotel, color in zip(['City Hotel', 'Resort Hotel'], ['#4472C4', '#ED7D31']):
    sub = adr_mes_hotel[adr_mes_hotel['hotel'] == hotel]
    axes[0].plot(sub['arrival_date_month'], sub['adr'],
                 marker='o', linewidth=2.5, label=hotel, color=color)
axes[0].set_title('ADR promedio por mes')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('ADR promedio (€)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot por mes — Resort Hotel (donde la estacionalidad es más fuerte)
df_resort = df[df['hotel'] == 'Resort Hotel'].copy()
df_resort['arrival_date_month'] = pd.Categorical(
    df_resort['arrival_date_month'], categories=orden_meses, ordered=True)
sns.boxplot(data=df_resort.sort_values('arrival_date_month'),
            x='arrival_date_month', y='adr',
            palette='YlOrRd', ax=axes[1],
            flierprops=dict(marker='.', markersize=2, alpha=0.3))
axes[1].set_title('Distribución de tarifa por mes — Resort Hotel')
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('ADR (€)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(0, df_resort['adr'].quantile(0.98))

plt.tight_layout()
plt.show()

**Interpretación:**

- **Resort Hotel muestra una estacionalidad muy marcada**: tarifas bajas de noviembre a febrero (temporada baja) y picos altos en julio-agosto (verano europeo). El ratio entre temporada alta y baja puede superar el 2x.
- **City Hotel es mucho más estable** durante el año, con leves alzas en primavera y otoño (temporada de congresos y turismo urbano), pero sin los picos extremos del resort.
- El boxplot del Resort Hotel confirma que no solo sube la mediana en verano — también aumenta la dispersión: los hoteles cobran tarifas muy variables según el tipo de habitación y demanda.
- **Implicación estratégica:** el Resort puede beneficiarse de precios dinámicos agresivos; el City necesita estrategias más uniformes.

---
## 6. Visualización multivariada #4 — Anticipación (lead_time) vs cancelación

In [ ]:
# Agrupamos lead_time en bins para ver tasa de cancelación por nivel de anticipación
bins = [-1, 7, 30, 90, 180, 365, 800]
labels = ['0-7 días', '8-30 días', '31-90 días', '91-180 días', '181-365 días', '>365 días']
df['lead_time_bin'] = pd.cut(df['lead_time'], bins=bins, labels=labels)

tasa_lead = (df.groupby(['lead_time_bin', 'hotel'])['is_canceled']
               .mean()
               .mul(100)
               .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Anticipación de la reserva y su efecto en la cancelación', fontsize=14, fontweight='bold')

# Barplot agrupado: tasa de cancelación por bin × hotel
sns.barplot(data=tasa_lead, x='lead_time_bin', y='is_canceled',
            hue='hotel', palette=['#4472C4', '#ED7D31'], ax=axes[0])
axes[0].set_title('Tasa de cancelación por anticipación')
axes[0].set_xlabel('Anticipación (lead_time)')
axes[0].set_ylabel('% canceladas')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Hotel')
axes[0].grid(axis='y', alpha=0.3)

# Boxplot: lead_time según si se canceló o no
sns.boxplot(data=df, x='is_canceled', y='lead_time',
            palette={0: '#70AD47', 1: '#C00000'}, ax=axes[1],
            flierprops=dict(marker='.', markersize=2, alpha=0.3))
axes[1].set_title('Distribución de anticipación según cancelación')
axes[1].set_xlabel('¿Canceló?')
axes[1].set_xticklabels(['No', 'Sí'])
axes[1].set_ylabel('Anticipación (días)')
axes[1].set_ylim(0, df['lead_time'].quantile(0.99))

plt.tight_layout()
plt.show()

**Interpretación:**

- **A mayor anticipación, mayor tasa de cancelación** — el patrón es nítido y consistente entre ambos hoteles. Las reservas hechas con más de 6 meses de antelación cancelan en más del 50% de los casos.
- Las reservas de **último minuto (0–7 días)** tienen la tasa más baja de cancelación: cuando alguien reserva para mañana, casi siempre cumple.
- El boxplot confirma: la mediana de `lead_time` de las reservas canceladas es notablemente mayor que la de las que se cumplen.
- **Insight de negocio:** la anticipación es un excelente predictor de cancelación. El hotel podría sobreventar de forma más agresiva en bookings de larga anticipación, sabiendo que estadísticamente muchos no llegarán.

---
## 7. Análisis adicional

### 7.1 Estadísticas descriptivas completas

In [ ]:
cols_num = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
            'adults', 'children', 'babies', 'total_nights',
            'previous_cancellations', 'previous_bookings_not_canceled',
            'booking_changes', 'days_in_waiting_list', 'adr',
            'required_car_parking_spaces', 'total_of_special_requests']

stats = df[cols_num].agg([
    'count', 'mean', 'median',
    lambda x: x.std(),
    lambda x: x.min(),
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75),
    lambda x: x.max(),
    lambda x: x.skew()
]).round(3)
stats.index = ['count', 'mean', 'median', 'std', 'min', 'Q1', 'Q3', 'max', 'skewness']

print('Estadísticas descriptivas — variables numéricas:')
display(stats)

print('\nTop 10 países por volumen de reservas:')
display(df['country'].value_counts().head(10))

### 7.2 Identificación de tendencias

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Tendencias del Mercado Hotelero', fontsize=15, fontweight='bold', y=1.01)

# 1. Reservas por mes (estacionalidad agregada)
orden_meses = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
reservas_mes = df['arrival_date_month'].value_counts().reindex(orden_meses)
axes[0, 0].bar(range(len(reservas_mes)), reservas_mes.values,
               color=sns.color_palette('viridis', len(reservas_mes)), edgecolor='white')
axes[0, 0].set_xticks(range(len(reservas_mes)))
axes[0, 0].set_xticklabels([m[:3] for m in reservas_mes.index], rotation=0)
axes[0, 0].set_title('Volumen de reservas por mes')
axes[0, 0].set_xlabel('Mes')
axes[0, 0].set_ylabel('Cantidad de reservas')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Top 10 países
top_paises = df['country'].value_counts().head(10)
axes[0, 1].barh(top_paises.index[::-1], top_paises.values[::-1],
                color='#4A90D9', edgecolor='white')
axes[0, 1].set_title('Top 10 países de origen')
axes[0, 1].set_xlabel('Cantidad de reservas')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Segmento de mercado vs cancelación
cancel_segmento = (df.groupby('market_segment')['is_canceled']
                     .agg(['mean', 'count'])
                     .sort_values('count', ascending=False)
                     .head(8))
cancel_segmento['mean'] *= 100
axes[1, 0].barh(cancel_segmento.index[::-1], cancel_segmento['mean'][::-1],
                color='#C00000', edgecolor='white', alpha=0.75)
axes[1, 0].set_title('Tasa de cancelación por segmento de mercado')
axes[1, 0].set_xlabel('% canceladas')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Duración de estancia: distribución
axes[1, 1].hist(df['total_nights'].clip(upper=14), bins=15,
                color='#7030A0', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('Duración de la estancia (recortada a 14 noches)')
axes[1, 1].set_xlabel('Noches totales')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].axvline(df['total_nights'].median(), color='red', linestyle='--',
                   linewidth=2, label=f"Mediana: {df['total_nights'].median():.0f} noches")
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**Tendencias observadas:**

1. **Estacionalidad clara**: julio y agosto concentran el volumen más alto de reservas (temporada alta de verano europeo). Los meses de noviembre, diciembre y enero son los más débiles.

2. **Dominio del mercado europeo**: Portugal lidera ampliamente (el dataset es de hoteles portugueses), seguido por Reino Unido, Francia, España y Alemania. El mercado se concentra en Europa occidental.

3. **Cancelación por segmento**: los segmentos **Groups** y **Online TA (travel agency)** tienen las tasas más altas de cancelación. **Direct** y **Corporate** son los más estables, lo que tiene sentido — el cliente que reserva directamente tiene mayor intención real.

4. **Estancias cortas dominan**: la mediana es de 2-3 noches. La mayoría de las reservas son de fines de semana o estancias cortas, consistente con la naturaleza turística del dataset.

---
## 8. Conclusiones y hallazgos principales

**Hallazgos de limpieza:**

- El dataset tenía un **27% de duplicados** (~32k filas), que sin eliminar habrían distorsionado todo el análisis.
- Los nulos están concentrados en `company` (94%) y `agent` (14%) — son códigos identificadores, no métricas, y se imputaron con 0 para indicar 'sin convenio/sin agencia'.
- Se detectaron y corrigieron: `adr` negativos, una tarifa extrema (5400), reservas sin huéspedes (0 adultos + 0 niños + 0 bebés), y la categoría `meal = 'Undefined'` unificada con `'SC'`.
- Las fechas se convirtieron a `datetime` real para permitir análisis temporales.

**Hallazgos del análisis:**

1. **El tipo de hotel define el comportamiento del cliente** — City Hotel tiene mayor volumen y mayor cancelación; Resort Hotel tiene menor volumen pero mayor variabilidad de tarifa por estacionalidad.

2. **Estacionalidad fuerte en Resort, leve en City** — el resort puede beneficiarse de pricing dinámico agresivo; el city necesita estrategias más uniformes durante el año.

3. **La anticipación predice la cancelación** — las reservas con más de 90 días de antelación cancelan más del 40-50% de las veces. Esto permite estrategias de overbooking calibrado.

4. **Los segmentos Groups y Online TA son los más volátiles** — alta cancelación, lo que indica que los términos de cancelación flexibles habilitados por las OTAs estimulan reservas que luego no se cumplen.

5. **El mercado es predominantemente europeo** — concentración fuerte en Portugal, Reino Unido, Francia y España. Cualquier estrategia de marketing internacional debería partir de esta base.